# Neural Network vs Graph Neural Network Comparison

## Research-Backed Methodology

This notebook implements a comprehensive, research-backed comparison between SimpleNN (feedforward MLP baseline) and GraphSAGE (GNN) models trained on surface code error correction. The comparison follows established ML benchmarking practices from Nature Scientific Reports, JMLR, ICLR/NeurIPS, and IEEE publications.

**Key Principles:**
- Fair comparison with parameter budget matching
- Statistical significance testing (not just mean accuracy)
- Identical evaluation protocols
- Multiple performance indices (accuracy, speed, memory, complexity)
- Proper baseline comparison (SimpleNN as structure-agnostic baseline)

**Distances Analyzed:** d=3, d=5, d=7, d=9, d=11, d=13

## 1. Imports and Setup

In [1]:
import sys
import json
import time
import gc
import os
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from scipy.stats import wilcoxon, chi2_contingency
import math

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/results/nn_vs_gnn_comparison -> code/

sys.path.insert(0, str(BASE_PATH))

# Import model classes
from benchmark_models import SimpleNN, SurfaceCodeSampler
from models import GraphSAGE, SparseGraph

# Set up paths
RESULTS_DIR = BASE_PATH / "results" / "nn_vs_gnn_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_RESULTS_DIR = RESULTS_DIR / "results"
OUTPUT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  OUTPUT_RESULTS_DIR: {OUTPUT_RESULTS_DIR}")
print(f"  PLOTS_DIR: {PLOTS_DIR}")

Using device: cuda
Using device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti

Paths:
  BASE_PATH: ..\..
  RESULTS_DIR: ..\..\results\nn_vs_gnn_comparison
  OUTPUT_RESULTS_DIR: ..\..\results\nn_vs_gnn_comparison\results
  PLOTS_DIR: ..\..\results\nn_vs_gnn_comparison\plots


## 2. Configuration

In [2]:
# Distances to compare
DISTANCES = [3, 5, 7, 9, 11, 13]

# Test dataset configuration
TEST_SAMPLES_PER_DISTANCE = 10000  # Per distance for evaluation
BENCHMARK_SAMPLES = 10000  # For inference speed benchmarking

# Inference benchmarking configuration
BATCH_SIZES = [1, 8, 16, 32, 64, 128, 256]
NUM_WARMUP_RUNS = 10  # Warmup batches before timing
NUM_BENCHMARK_RUNS = 5  # Number of independent runs for statistical robustness
NUM_ACCURACY_RUNS = 4  # Number of independent accuracy evaluations

# Statistical test significance level
ALPHA = 0.05

# Random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Configuration:")
print(f"  Distances: {DISTANCES}")
print(f"  Test samples per distance: {TEST_SAMPLES_PER_DISTANCE:,}")
print(f"  Benchmark samples: {BENCHMARK_SAMPLES:,}")
print(f"  Batch sizes for benchmarking: {BATCH_SIZES}")
print(f"  Number of benchmark runs: {NUM_BENCHMARK_RUNS}")
print(f"  Number of accuracy runs: {NUM_ACCURACY_RUNS}")
print(f"  Significance level (alpha): {ALPHA}")
print(f"  Random seed: {SEED}")

Configuration:
  Distances: [3, 5, 7, 9, 11, 13]
  Test samples per distance: 10,000
  Benchmark samples: 10,000
  Batch sizes for benchmarking: [1, 8, 16, 32, 64, 128, 256]
  Number of benchmark runs: 5
  Number of accuracy runs: 4
  Significance level (alpha): 0.05
  Random seed: 42


## 3. Helper Functions

In [3]:
def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_model_size_mb(model_path: Path) -> float:
    """Get model file size in MB."""
    if model_path.exists():
        return model_path.stat().st_size / (1024 * 1024)
    return 0.0


def load_simplenn_model(distance: int, base_path: Path) -> Tuple[SimpleNN, Dict]:
    """Load SimpleNN model for a given distance."""
    # Use revised_training folder
    model_path = base_path / "nn" / "training" / "models" / "revised_training" / f"d{distance}_specialist.pt"
    
    if not model_path.exists():
        raise FileNotFoundError(f"SimpleNN model not found for d={distance} at {model_path}")
    
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    # Get num_detectors from checkpoint or calculate
    if 'num_detectors' in checkpoint:
        num_detectors = checkpoint['num_detectors']
    else:
        # Calculate from distance
        import stim
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            rounds=distance,
            distance=distance,
            after_clifford_depolarization=0.001,
        )
        num_detectors = circuit.num_detectors
    
    # Get hyperparameters
    if 'hyperparams' in checkpoint:
        hyperparams = checkpoint['hyperparams']
        hidden_dims = tuple(hyperparams.get('hidden_dims', [512, 1024, 2048]))
        dropout = hyperparams.get('dropout', 0.1)
    else:
        hidden_dims = (512, 1024, 2048)
        dropout = 0.1
    
    # Initialize and load model
    model = SimpleNN(
        nickname=f"d{distance}_specialist",
        in_channels=num_detectors,
        hidden_dims=hidden_dims,
        dropout=dropout,
        device=device,
        base_path=base_path
    )
    
    if 'state_dict' in checkpoint:
        # Use strict=False to handle architecture differences (older models without LayerNorm)
        missing_keys, unexpected_keys = model.model.load_state_dict(checkpoint['state_dict'], strict=False)
        if missing_keys or unexpected_keys:
            print(f"    Note: Architecture mismatch detected for d={distance}")
            print(f"    Missing keys: {len(missing_keys)}, Unexpected keys: {len(unexpected_keys)}")
    else:
        # Try loading from model's own state
        model.load(str(model_path))
    
    model.model.eval()
    
    # Count parameters
    num_params = count_parameters(model.model)
    model_size_mb = get_model_size_mb(model_path)
    
    info = {
        'distance': distance,
        'num_detectors': num_detectors,
        'num_parameters': num_params,
        'model_size_mb': model_size_mb,
        'model_path': str(model_path),
        'hyperparams': hyperparams if 'hyperparams' in checkpoint else {}
    }
    
    return model, info


def load_graphsage_model(distance: int, base_path: Path) -> Tuple[GraphSAGE, Dict]:
    """Load GraphSAGE model for a given distance."""
    # Use revised_training folder
    model_path = base_path / "gSAGE" / "distances" / "models" / "revised_training" / f"d{distance}.pt"
    
    if not model_path.exists():
        raise FileNotFoundError(f"GraphSAGE model not found for d={distance} at {model_path}")
    
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    # Get config
    if 'config' in checkpoint:
        config = checkpoint['config']
    else:
        # Default config
        config = {
            'in_channels': 5,
            'hidden_dim': 128,
            'num_layers': 5,
            'dropout': 0.0,
            'aggr': 'max'
        }
    
    # Initialize and load model
    model = GraphSAGE(
        nickname=f"d{distance}",
        in_channels=config.get('in_channels', 5),
        hidden_dim=config.get('hidden_dim', 128),
        num_layers=config.get('num_layers', 5),
        dropout=config.get('dropout', 0.0),
        aggr=config.get('aggr', 'max'),
        device=device,
        base_path=base_path
    )
    
    if 'state_dict' in checkpoint:
        model.model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load(str(model_path))
    
    model.model.eval()
    
    # Count parameters
    num_params = count_parameters(model.model)
    model_size_mb = get_model_size_mb(model_path)
    
    info = {
        'distance': distance,
        'num_parameters': num_params,
        'model_size_mb': model_size_mb,
        'model_path': str(model_path),
        'config': config
    }
    
    return model, info


print("Helper functions defined.")

Helper functions defined.


In [4]:
# Load all models
simplenn_models = {}
graphsage_models = {}
simplenn_info = {}
graphsage_info = {}

print("Loading SimpleNN models...")
for d in DISTANCES:
    try:
        model, info = load_simplenn_model(d, BASE_PATH)
        simplenn_models[d] = model
        simplenn_info[d] = info
        print(f"  ✓ d={d}: {info['num_parameters']:,} params, {info['model_size_mb']:.2f} MB")
    except Exception as e:
        print(f"  ✗ d={d}: Failed to load - {e}")

print("\nLoading GraphSAGE models...")
for d in DISTANCES:
    try:
        model, info = load_graphsage_model(d, BASE_PATH)
        graphsage_models[d] = model
        graphsage_info[d] = info
        print(f"  ✓ d={d}: {info['num_parameters']:,} params, {info['model_size_mb']:.2f} MB")
    except Exception as e:
        print(f"  ✗ d={d}: Failed to load - {e}")

print(f"\nLoaded {len(simplenn_models)} SimpleNN models and {len(graphsage_models)} GraphSAGE models")

Loading SimpleNN models...
SimpleNN initialized: SimpleNN(nickname='d3_specialist', in_channels=24, hidden_dims=(512, 1024, 2048), dropout=0.1)
  ✗ d=3: Failed to load - Error(s) in loading state_dict for SimpleNNModel:
	size mismatch for layers.9.weight: copying a param with shape torch.Size([1, 2048]) from checkpoint, the shape in current model is torch.Size([2048]).
	size mismatch for layers.9.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([2048]).
SimpleNN initialized: SimpleNN(nickname='d5_specialist', in_channels=120, hidden_dims=(512, 1024, 2048), dropout=0.1)
  ✗ d=5: Failed to load - Error(s) in loading state_dict for SimpleNNModel:
	size mismatch for layers.9.weight: copying a param with shape torch.Size([1, 2048]) from checkpoint, the shape in current model is torch.Size([2048]).
	size mismatch for layers.9.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([20

## 5. Load Training Results

In [5]:
# Load SimpleNN training results from revised_training folder
simplenn_training_results = {}
simplenn_results_path = BASE_PATH / "nn" / "training" / "results" / "revised_training" / "results.json"

if simplenn_results_path.exists():
    with open(simplenn_results_path, 'r') as f:
        simplenn_results = json.load(f)
    
    for result in simplenn_results:
        # Include both 'completed' and 'completed blowup' status
        if result.get('status', '').startswith('completed'):
            d = result['distance']
            simplenn_training_results[d] = result
    print(f"Loaded SimpleNN training results for {len(simplenn_training_results)} distances")
else:
    print(f"Warning: SimpleNN results file not found at {simplenn_results_path}")

# Load GraphSAGE training results from revised_training folder
graphsage_training_results = {}
for d in DISTANCES:
    results_path = BASE_PATH / "gSAGE" / "distances" / "results" / "revised_training" / f"d{d}_training.json"
    if results_path.exists():
        with open(results_path, 'r') as f:
            graphsage_training_results[d] = json.load(f)
    else:
        print(f"Warning: GraphSAGE results file not found for d={d} at {results_path}")

print(f"Loaded GraphSAGE training results for {len(graphsage_training_results)} distances")

Loaded SimpleNN training results for 6 distances
Loaded GraphSAGE training results for 6 distances


## 6. Parameter Budget Analysis

In [6]:
# Create parameter comparison table
param_comparison = []
for d in DISTANCES:
    if d in simplenn_info and d in graphsage_info:
        param_comparison.append({
            'distance': d,
            'simplenn_params': simplenn_info[d]['num_parameters'],
            'graphsage_params': graphsage_info[d]['num_parameters'],
            'simplenn_size_mb': simplenn_info[d]['model_size_mb'],
            'graphsage_size_mb': graphsage_info[d]['model_size_mb'],
            'param_ratio': graphsage_info[d]['num_parameters'] / simplenn_info[d]['num_parameters'] if simplenn_info[d]['num_parameters'] > 0 else 0
        })

param_df = pd.DataFrame(param_comparison)
print("Parameter Budget Comparison:")
print(param_df.to_string(index=False))

# Calculate statistics
if len(param_comparison) > 0:
    avg_ratio = param_df['param_ratio'].mean()
    print(f"\nAverage parameter ratio (GraphSAGE/SimpleNN): {avg_ratio:.2f}x")
    print(f"SimpleNN avg params: {param_df['simplenn_params'].mean():,.0f}")
    print(f"GraphSAGE avg params: {param_df['graphsage_params'].mean():,.0f}")
    
    if avg_ratio > 1.5 or avg_ratio < 0.67:
        print(f"\n⚠️  WARNING: Significant parameter budget difference ({avg_ratio:.2f}x)")
        print("   This should be documented in the comparison.")
    else:
        print(f"\n✓ Parameter budgets are reasonably matched ({avg_ratio:.2f}x)")

Parameter Budget Comparison:
 distance  simplenn_params  graphsage_params  simplenn_size_mb  graphsage_size_mb  param_ratio
        9          3002881            142593         34.385299           0.563184     0.047485
       11          3310081            142593         37.900983           1.667798     0.043078
       13          3752449            142593         42.963483           1.667798     0.038000

Average parameter ratio (GraphSAGE/SimpleNN): 0.04x
SimpleNN avg params: 3,355,137
GraphSAGE avg params: 142,593

⚠️  WARNING: Significant parameter budget difference (0.04x)
   This should be documented in the comparison.


## 7. Generate Shared Test Sets (Fair Comparison Protocol)

In [7]:
# Generate shared test datasets for fair comparison
# Same random seed ensures identical data for both models
shared_test_data = {}
graph_builder = SparseGraph(k_neighbors=6, device=torch.device('cpu'))
sampler = SurfaceCodeSampler(p=0.005, device=torch.device('cpu'))

print("Generating shared test datasets...")
for d in DISTANCES:
    if d not in simplenn_models or d not in graphsage_models:
        print(f"  Skipping d={d} (models not loaded)")
        continue
    
    # Generate detections with fixed seed for reproducibility
    torch.manual_seed(SEED + d)  # Different seed per distance but deterministic
    np.random.seed(SEED + d)
    
    detections, labels = sampler.sample(
        d=d,
        num_samples=TEST_SAMPLES_PER_DISTANCE,
        p_values=[0.001, 0.003, 0.005, 0.007],
        p_weights=[0.25, 0.25, 0.25, 0.25]
    )
    
    # Convert to graphs for GraphSAGE
    graphs = graph_builder.batch_to_pyg(detections, labels)
    
    shared_test_data[d] = {
        'detections': detections.cpu(),  # For SimpleNN
        'labels': labels.cpu(),
        'graphs': graphs,  # For GraphSAGE
        'num_samples': len(labels)
    }
    
    print(f"  ✓ d={d}: {len(labels):,} samples")

print(f"\nGenerated shared test data for {len(shared_test_data)} distances")

SparseGraph initialized:
  K neighbors: 6
  Device: cpu
  Mode: Dynamic (supports any code distance)
SurfaceCodeSampler initialized:
  Default error rate: 0.005
  Device: cpu
  Mode: Dynamic (supports any code distance)
Generating shared test datasets...
  Skipping d=3 (models not loaded)
  Skipping d=5 (models not loaded)
  Skipping d=7 (models not loaded)
  ✓ d=9: 10,000 samples
  ✓ d=11: 10,000 samples
  ✓ d=13: 10,000 samples

Generated shared test data for 3 distances


## 8. Inference Speed Benchmarking

In [ ]:
def benchmark_inference_speed(
    model,
    data,
    batch_sizes: List[int],
    num_warmup: int = 10,
    num_runs: int = 5,
    is_graph_model: bool = False
) -> Dict:
    """
    Benchmark inference speed with multiple runs for statistical robustness.
    
    Returns:
        Dict with mean ± std for each batch size
    """
    model.model.eval()
    model.model.to(device)
    
    results = {}
    
    for batch_size in batch_sizes:
        if is_graph_model:
            # GraphSAGE: use DataLoader
            loader = DataLoader(data, batch_size=batch_size, shuffle=False)
            num_samples = len(data)
        else:
            # SimpleNN: use tensor batches
            num_samples = len(data)
        
        # Warmup
        with torch.no_grad():
            warmup_count = 0
            if is_graph_model:
                for batch in loader:
                    if warmup_count >= num_warmup:
                        break
                    batch = batch.to(device)
                    _ = model.model(batch)
                    warmup_count += 1
            else:
                for i in range(0, min(num_warmup * batch_size, num_samples), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    _ = model.model(batch)
                    warmup_count += 1
        
        # Synchronize GPU
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        # Timed runs
        run_times = []
        for run in range(num_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            start_time = time.time()
            
            with torch.no_grad():
                if is_graph_model:
                    for batch in loader:
                        batch = batch.to(device)
                        _ = model.model(batch)
                else:
                    for i in range(0, num_samples, batch_size):
                        batch = data[i:i+batch_size].to(device)
                        _ = model.model(batch)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            elapsed = time.time() - start_time
            run_times.append(elapsed)
        
        # Calculate statistics
        mean_time = np.mean(run_times)
        std_time = np.std(run_times)
        throughput = num_samples / mean_time  # samples per second
        latency_per_sample = (mean_time / num_samples) * 1e6  # microseconds
        
        results[batch_size] = {
            'mean_time_seconds': mean_time,
            'std_time_seconds': std_time,
            'throughput_samples_per_sec': throughput,
            'latency_per_sample_us': latency_per_sample,
            'runs': run_times
        }
    
    return results


# Benchmark all models
print("Benchmarking inference speed...")
inference_benchmarks = {}

for d in DISTANCES:
    if d not in shared_test_data:
        continue
    
    inference_benchmarks[d] = {}
    
    # Benchmark SimpleNN
    if d in simplenn_models:
        print(f"  Benchmarking SimpleNN d={d}...")
        detections = shared_test_data[d]['detections']
        results = benchmark_inference_speed(
            simplenn_models[d],
            detections,
            BATCH_SIZES,
            NUM_WARMUP_RUNS,
            NUM_BENCHMARK_RUNS,
            is_graph_model=False
        )
        inference_benchmarks[d]['simplenn'] = results
    
    # Benchmark GraphSAGE
    if d in graphsage_models:
        print(f"  Benchmarking GraphSAGE d={d}...")
        graphs = shared_test_data[d]['graphs']
        results = benchmark_inference_speed(
            graphsage_models[d],
            graphs,
            BATCH_SIZES,
            NUM_WARMUP_RUNS,
            NUM_BENCHMARK_RUNS,
            is_graph_model=True
        )
        inference_benchmarks[d]['graphsage'] = results

print("\n✓ Inference benchmarking complete")

Benchmarking inference speed...
  Benchmarking SimpleNN d=9...
  Benchmarking GraphSAGE d=9...
  Benchmarking SimpleNN d=11...
  Benchmarking GraphSAGE d=11...


In [ ]:
def evaluate_accuracy_metrics(
    model,
    data,
    labels,
    num_runs: int = 4,
    threshold: float = 0.5,
    is_graph_model: bool = False,
    batch_size: int = 256
) -> Dict:
    """
    Evaluate accuracy metrics with multiple runs for statistical robustness.
    
    Returns:
        Dict with mean ± std for accuracy, precision, recall, F1
    """
    model.model.eval()
    model.model.to(device)
    
    all_accuracies = []
    all_precisions = []
    all_recalls = []
    all_f1s = []
    all_predictions = []
    
    for run in range(num_runs):
        with torch.no_grad():
            predictions = []
            
            if is_graph_model:
                # GraphSAGE: use DataLoader
                loader = DataLoader(data, batch_size=batch_size, shuffle=False)
                for batch in loader:
                    batch = batch.to(device)
                    pred = model.model(batch)
                    predictions.append(pred.cpu())
            else:
                # SimpleNN: use tensor batches
                for i in range(0, len(data), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    pred = model.model(batch)
                    predictions.append(pred.cpu())
            
            predictions = torch.cat(predictions, dim=0).squeeze()
            binary_preds = (predictions >= threshold).float()
            labels_tensor = labels.float()
            
            # Calculate metrics
            correct = (binary_preds == labels_tensor).sum().item()
            accuracy = correct / len(labels_tensor)
            
            # Precision, Recall, F1
            tp = ((binary_preds == 1) & (labels_tensor == 1)).sum().item()
            fp = ((binary_preds == 1) & (labels_tensor == 0)).sum().item()
            fn = ((binary_preds == 0) & (labels_tensor == 1)).sum().item()
            tn = ((binary_preds == 0) & (labels_tensor == 0)).sum().item()
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
            
            all_accuracies.append(accuracy)
            all_precisions.append(precision)
            all_recalls.append(recall)
            all_f1s.append(f1)
            
            if run == 0:  # Store predictions from first run for McNemar's test
                all_predictions = binary_preds.numpy()
    
    # Calculate mean ± std
    mean_acc = np.mean(all_accuracies)
    std_acc = np.std(all_accuracies)
    mean_prec = np.mean(all_precisions)
    std_prec = np.std(all_precisions)
    mean_rec = np.mean(all_recalls)
    std_rec = np.std(all_recalls)
    mean_f1 = np.mean(all_f1s)
    std_f1 = np.std(all_f1s)
    
    # 95% CI
    ci_acc = 1.96 * std_acc / np.sqrt(num_runs)
    ci_f1 = 1.96 * std_f1 / np.sqrt(num_runs)
    
    return {
        'accuracy_mean': mean_acc,
        'accuracy_std': std_acc,
        'accuracy_ci_95': ci_acc,
        'precision_mean': mean_prec,
        'precision_std': std_prec,
        'recall_mean': mean_rec,
        'recall_std': std_rec,
        'f1_mean': mean_f1,
        'f1_std': std_f1,
        'f1_ci_95': ci_f1,
        'all_accuracies': all_accuracies,
        'all_f1s': all_f1s,
        'predictions': all_predictions  # For McNemar's test
    }


# Evaluate all models
print("Evaluating accuracy metrics...")
accuracy_results = {}

for d in DISTANCES:
    if d not in shared_test_data:
        continue
    
    accuracy_results[d] = {}
    labels = shared_test_data[d]['labels']
    
    # Evaluate SimpleNN
    if d in simplenn_models:
        print(f"  Evaluating SimpleNN d={d}...")
        detections = shared_test_data[d]['detections']
        results = evaluate_accuracy_metrics(
            simplenn_models[d],
            detections,
            labels,
            NUM_ACCURACY_RUNS,
            is_graph_model=False
        )
        accuracy_results[d]['simplenn'] = results
    
    # Evaluate GraphSAGE
    if d in graphsage_models:
        print(f"  Evaluating GraphSAGE d={d}...")
        graphs = shared_test_data[d]['graphs']
        results = evaluate_accuracy_metrics(
            graphsage_models[d],
            graphs,
            labels,
            NUM_ACCURACY_RUNS,
            is_graph_model=True
        )
        accuracy_results[d]['graphsage'] = results

print("\n✓ Accuracy evaluation complete")

Evaluating accuracy metrics...

✓ Accuracy evaluation complete


In [ ]:
def wilcoxon_test(simplenn_accs: List[float], graphsage_accs: List[float]) -> Dict:
    """Perform Wilcoxon signed-rank test."""
    statistic, p_value = wilcoxon(simplenn_accs, graphsage_accs, alternative='two-sided')
    
    # Effect size (rank-biserial correlation)
    n = len(simplenn_accs)
    z = stats.norm.ppf(p_value / 2) if p_value > 0 else 0
    r = abs(z) / np.sqrt(n)
    
    return {
        'test': 'Wilcoxon signed-rank',
        'statistic': float(statistic),
        'p_value': float(p_value),
        'significant': p_value < ALPHA,
        'effect_size_r': float(r),
        'interpretation': 'large' if r > 0.5 else ('medium' if r > 0.3 else 'small')
    }


def mcnemar_test(simplenn_preds: np.ndarray, graphsage_preds: np.ndarray, labels: np.ndarray) -> Dict:
    """Perform McNemar's test."""
    # Create contingency table
    # Both correct, SimpleNN wrong, GraphSAGE wrong, Both wrong
    both_correct = ((simplenn_preds == labels) & (graphsage_preds == labels)).sum()
    simplenn_wrong = ((simplenn_preds != labels) & (graphsage_preds == labels)).sum()
    graphsage_wrong = ((simplenn_preds == labels) & (graphsage_preds != labels)).sum()
    both_wrong = ((simplenn_preds != labels) & (graphsage_preds != labels)).sum()
    
    # McNemar's test (using chi-square approximation)
    # Only use discordant pairs
    b = simplenn_wrong
    c = graphsage_wrong
    
    if b + c == 0:
        # No discordant pairs
        return {
            'test': "McNemar's",
            'statistic': 0.0,
            'p_value': 1.0,
            'significant': False,
            'note': 'No discordant pairs'
        }
    
    chi2 = (abs(b - c) - 1)**2 / (b + c)  # Continuity correction
    p_value = 1 - stats.chi2.cdf(chi2, df=1)
    
    return {
        'test': "McNemar's",
        'statistic': float(chi2),
        'p_value': float(p_value),
        'significant': p_value < ALPHA,
        'discordant_pairs': int(b + c),
        'simplenn_wrong_only': int(b),
        'graphsage_wrong_only': int(c)
    }


def cohens_d(group1: List[float], group2: List[float]) -> float:
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = np.mean(group1), np.mean(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    d = (mean1 - mean2) / pooled_std
    return float(d)


# Perform statistical tests
print("Performing statistical tests...")
statistical_tests = {}

# Wilcoxon test across all distances
simplenn_all_accs = []
graphsage_all_accs = []

for d in DISTANCES:
    if d not in accuracy_results:
        continue
    
    if 'simplenn' in accuracy_results[d] and 'graphsage' in accuracy_results[d]:
        simplenn_accs = accuracy_results[d]['simplenn']['all_accuracies']
        graphsage_accs = accuracy_results[d]['graphsage']['all_accuracies']
        
        simplenn_all_accs.extend(simplenn_accs)
        graphsage_all_accs.extend(graphsage_accs)
        
        # Per-distance Wilcoxon test
        wilcoxon_result = wilcoxon_test(simplenn_accs, graphsage_accs)
        
        # McNemar's test (on first run predictions)
        labels_np = shared_test_data[d]['labels'].numpy()
        simplenn_preds = accuracy_results[d]['simplenn']['predictions']
        graphsage_preds = accuracy_results[d]['graphsage']['predictions']
        mcnemar_result = mcnemar_test(simplenn_preds, graphsage_preds, labels_np)
        
        # Effect size (Cohen's d)
        cohens_d_val = cohens_d(simplenn_accs, graphsage_accs)
        
        statistical_tests[d] = {
            'wilcoxon': wilcoxon_result,
            'mcnemar': mcnemar_result,
            'cohens_d': float(cohens_d_val),
            'effect_size_interpretation': 'large' if abs(cohens_d_val) > 0.8 else ('medium' if abs(cohens_d_val) > 0.5 else 'small')
        }

# Overall Wilcoxon test
if len(simplenn_all_accs) > 0 and len(graphsage_all_accs) > 0:
    overall_wilcoxon = wilcoxon_test(simplenn_all_accs, graphsage_all_accs)
    overall_cohens_d = cohens_d(simplenn_all_accs, graphsage_all_accs)
    
    statistical_tests['overall'] = {
        'wilcoxon': overall_wilcoxon,
        'cohens_d': float(overall_cohens_d),
        'effect_size_interpretation': 'large' if abs(overall_cohens_d) > 0.8 else ('medium' if abs(overall_cohens_d) > 0.5 else 'small')
    }

print("✓ Statistical testing complete")
print(f"\nOverall Wilcoxon test: p={statistical_tests.get('overall', {}).get('wilcoxon', {}).get('p_value', 'N/A'):.4f}")

Performing statistical tests...
✓ Statistical testing complete


ValueError: Unknown format code 'f' for object of type 'str'

## 11. Training Convergence Analysis

In [ ]:
# Extract training convergence data
training_convergence = {}

for d in DISTANCES:
    training_convergence[d] = {}
    
    # SimpleNN training data
    if d in simplenn_training_results:
        result = simplenn_training_results[d]
        if 'epoch_metrics' in result:
            epoch_metrics = result['epoch_metrics']
            training_convergence[d]['simplenn'] = {
                'epochs': epoch_metrics.get('epoch', []),
                'train_loss': epoch_metrics.get('train_loss', []),
                'val_accuracy': epoch_metrics.get('val_accuracy', []),
                'train_accuracy': epoch_metrics.get('train_accuracy', []),
                'training_time_seconds': result.get('training_time_seconds', 0),
                'final_test_accuracy': result.get('test_accuracy', 0)
            }
        else:
            # Fallback to basic info
            training_convergence[d]['simplenn'] = {
                'final_test_accuracy': result.get('test_accuracy', 0),
                'training_time_seconds': result.get('training_time_seconds', 0),
                'epochs': result.get('epochs', 0)
            }
    
    # GraphSAGE training data
    if d in graphsage_training_results:
        result = graphsage_training_results[d]
        if 'epoch_metrics' in result:
            epoch_metrics = result['epoch_metrics']
            training_convergence[d]['graphsage'] = {
                'epochs': epoch_metrics.get('epochs', []),
                'train_loss': epoch_metrics.get('train_loss', []),
                'val_accuracy': epoch_metrics.get('val_accuracy', []),
                'train_accuracy': epoch_metrics.get('train_accuracy', []),
                'training_time_seconds': result.get('training_config', {}).get('epochs', 0) * (
                    epoch_metrics.get('epoch_time_seconds', [0])[0] if epoch_metrics.get('epoch_time_seconds') else 0
                ),
                'final_test_accuracy': result.get('final_metrics', {}).get('test_accuracy', 0) if 'final_metrics' in result else 0
            }
        else:
            training_convergence[d]['graphsage'] = {
                'final_test_accuracy': 0,
                'training_time_seconds': 0,
                'epochs': result.get('training_config', {}).get('epochs', 0)
            }

print("Training convergence data extracted")

## 12. Visualization Plots

In [ ]:
def plot_accuracy_vs_distance(accuracy_results, statistical_tests, output_path):
    """Graph 1: Accuracy vs Code Distance with error bars."""
    distances = sorted([d for d in accuracy_results.keys() if 'simplenn' in accuracy_results[d] and 'graphsage' in accuracy_results[d]])
    
    simplenn_accs = []
    simplenn_cis = []
    graphsage_accs = []
    graphsage_cis = []
    significance = []
    
    for d in distances:
        simplenn_accs.append(accuracy_results[d]['simplenn']['accuracy_mean'])
        simplenn_cis.append(accuracy_results[d]['simplenn']['accuracy_ci_95'])
        graphsage_accs.append(accuracy_results[d]['graphsage']['accuracy_mean'])
        graphsage_cis.append(accuracy_results[d]['graphsage']['accuracy_ci_95'])
        
        # Check significance
        if d in statistical_tests:
            sig = statistical_tests[d]['wilcoxon']['significant']
            significance.append('*' if sig else '')
        else:
            significance.append('')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(distances))
    width = 0.35
    
    # Plot with error bars
    bars1 = ax.errorbar(x - width/2, simplenn_accs, yerr=simplenn_cis, 
                       fmt='o-', label='SimpleNN', color='steelblue', 
                       capsize=5, capthick=2, linewidth=2, markersize=8)
    bars2 = ax.errorbar(x + width/2, graphsage_accs, yerr=graphsage_cis,
                       fmt='s-', label='GraphSAGE', color='coral',
                       capsize=5, capthick=2, linewidth=2, markersize=8)
    
    # Add significance markers
    for i, (d, sig) in enumerate(zip(distances, significance)):
        if sig:
            max_acc = max(simplenn_accs[i] + simplenn_cis[i], graphsage_accs[i] + graphsage_cis[i])
            ax.text(i, max_acc + 0.02, sig, ha='center', fontsize=14, fontweight='bold')
    
    ax.set_xlabel('Code Distance (d)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title('Accuracy vs Code Distance (95% CI)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'd={d}' for d in distances])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1.1])
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_pareto_frontier(accuracy_results, inference_benchmarks, output_path):
    """Graph 2: Pareto Frontier (Accuracy-Speed Tradeoff)."""
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Use batch size 64 for latency (representative)
    batch_size = 64
    
    for d in sorted(accuracy_results.keys()):
        if 'simplenn' not in accuracy_results[d] or 'graphsage' not in accuracy_results[d]:
            continue
        
        if d not in inference_benchmarks:
            continue
        
        # SimpleNN
        if 'simplenn' in inference_benchmarks[d] and batch_size in inference_benchmarks[d]['simplenn']:
            simplenn_latency = inference_benchmarks[d]['simplenn'][batch_size]['latency_per_sample_us']
            simplenn_acc = accuracy_results[d]['simplenn']['accuracy_mean']
            ax.scatter(simplenn_latency, simplenn_acc, s=200, marker='o', 
                      color='steelblue', alpha=0.7, edgecolors='black', linewidth=1.5)
            ax.annotate(f'SNN d={d}', (simplenn_latency, simplenn_acc), 
                       xytext=(5, 5), textcoords='offset points', fontsize=9)
        
        # GraphSAGE
        if 'graphsage' in inference_benchmarks[d] and batch_size in inference_benchmarks[d]['graphsage']:
            graphsage_latency = inference_benchmarks[d]['graphsage'][batch_size]['latency_per_sample_us']
            graphsage_acc = accuracy_results[d]['graphsage']['accuracy_mean']
            ax.scatter(graphsage_latency, graphsage_acc, s=200, marker='s',
                      color='coral', alpha=0.7, edgecolors='black', linewidth=1.5)
            ax.annotate(f'GS d={d}', (graphsage_latency, graphsage_acc),
                       xytext=(5, -15), textcoords='offset points', fontsize=9)
    
    ax.set_xlabel('Inference Latency (μs per sample)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title('Pareto Frontier: Accuracy vs Inference Latency', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='steelblue', edgecolor='black', label='SimpleNN'),
        Patch(facecolor='coral', edgecolor='black', label='GraphSAGE')
    ]
    ax.legend(handles=legend_elements, fontsize=11)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_convergence_curves(training_convergence, output_path):
    """Graph 3: Training Convergence Curves."""
    # Find distances with both models and epoch data
    distances_with_data = []
    for d in DISTANCES:
        if d in training_convergence:
            has_simplenn = 'simplenn' in training_convergence[d] and 'epochs' in training_convergence[d]['simplenn']
            has_graphsage = 'graphsage' in training_convergence[d] and 'epochs' in training_convergence[d]['graphsage']
            if has_simplenn or has_graphsage:
                distances_with_data.append(d)
    
    if len(distances_with_data) == 0:
        print("No convergence data available for plotting")
        return
    
    n_distances = len(distances_with_data)
    fig, axes = plt.subplots(1, min(n_distances, 3), figsize=(5*min(n_distances, 3), 5))
    if n_distances == 1:
        axes = [axes]
    
    for idx, d in enumerate(distances_with_data[:3]):  # Plot up to 3 distances
        ax = axes[idx] if n_distances > 1 else axes[0]
        
        # SimpleNN
        if 'simplenn' in training_convergence[d] and 'epochs' in training_convergence[d]['simplenn']:
            epochs = training_convergence[d]['simplenn']['epochs']
            val_acc = training_convergence[d]['simplenn'].get('val_accuracy', [])
            if len(val_acc) > 0:
                ax.plot(epochs, val_acc, 'o-', label='SimpleNN', color='steelblue', linewidth=2, markersize=4)
        
        # GraphSAGE
        if 'graphsage' in training_convergence[d] and 'epochs' in training_convergence[d]['graphsage']:
            epochs = training_convergence[d]['graphsage']['epochs']
            val_acc = training_convergence[d]['graphsage'].get('val_accuracy', [])
            if len(val_acc) > 0:
                ax.plot(epochs, val_acc, 's-', label='GraphSAGE', color='coral', linewidth=2, markersize=4)
        
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Validation Accuracy', fontsize=11)
        ax.set_title(f'Training Convergence - d={d}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    plt.suptitle('Training Convergence Comparison', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_throughput_vs_batchsize(inference_benchmarks, output_path):
    """Graph 4: Inference Throughput vs Batch Size."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    batch_sizes = sorted(BATCH_SIZES)
    
    # Aggregate across distances
    simplenn_throughputs = {bs: [] for bs in batch_sizes}
    graphsage_throughputs = {bs: [] for bs in batch_sizes}
    
    for d in inference_benchmarks.keys():
        if 'simplenn' in inference_benchmarks[d]:
            for bs in batch_sizes:
                if bs in inference_benchmarks[d]['simplenn']:
                    simplenn_throughputs[bs].append(inference_benchmarks[d]['simplenn'][bs]['throughput_samples_per_sec'])
        
        if 'graphsage' in inference_benchmarks[d]:
            for bs in batch_sizes:
                if bs in inference_benchmarks[d]['graphsage']:
                    graphsage_throughputs[bs].append(inference_benchmarks[d]['graphsage'][bs]['throughput_samples_per_sec'])
    
    # Calculate mean and std
    simplenn_means = [np.mean(simplenn_throughputs[bs]) if simplenn_throughputs[bs] else 0 for bs in batch_sizes]
    simplenn_stds = [np.std(simplenn_throughputs[bs]) if simplenn_throughputs[bs] else 0 for bs in batch_sizes]
    graphsage_means = [np.mean(graphsage_throughputs[bs]) if graphsage_throughputs[bs] else 0 for bs in batch_sizes]
    graphsage_stds = [np.std(graphsage_throughputs[bs]) if graphsage_throughputs[bs] else 0 for bs in batch_sizes]
    
    ax.errorbar(batch_sizes, simplenn_means, yerr=simplenn_stds, 
               fmt='o-', label='SimpleNN', color='steelblue', 
               capsize=5, capthick=2, linewidth=2, markersize=8)
    ax.errorbar(batch_sizes, graphsage_means, yerr=graphsage_stds,
               fmt='s-', label='GraphSAGE', color='coral',
               capsize=5, capthick=2, linewidth=2, markersize=8)
    
    ax.set_xlabel('Batch Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('Throughput (samples/second)', fontsize=12, fontweight='bold')
    ax.set_title('Inference Throughput vs Batch Size', fontsize=14, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xticks(batch_sizes)
    ax.set_xticklabels([str(bs) for bs in batch_sizes])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


def plot_critical_difference(accuracy_results, statistical_tests, output_path):
    """Graph 5: Critical Difference Diagram."""
    # Rank models by accuracy across distances
    distances = sorted([d for d in accuracy_results.keys() if 'simplenn' in accuracy_results[d] and 'graphsage' in accuracy_results[d]])
    
    model_ranks = {'SimpleNN': [], 'GraphSAGE': []}
    
    for d in distances:
        simplenn_acc = accuracy_results[d]['simplenn']['accuracy_mean']
        graphsage_acc = accuracy_results[d]['graphsage']['accuracy_mean']
        
        # Rank: lower rank = better (higher accuracy)
        if simplenn_acc > graphsage_acc:
            model_ranks['SimpleNN'].append(1)
            model_ranks['GraphSAGE'].append(2)
        elif graphsage_acc > simplenn_acc:
            model_ranks['SimpleNN'].append(2)
            model_ranks['GraphSAGE'].append(1)
        else:
            model_ranks['SimpleNN'].append(1.5)
            model_ranks['GraphSAGE'].append(1.5)
    
    avg_ranks = {model: np.mean(ranks) for model, ranks in model_ranks.items()}
    
    fig, ax = plt.subplots(figsize=(8, 4))
    
    models = list(avg_ranks.keys())
    ranks = [avg_ranks[m] for m in models]
    
    # Plot ranked bars
    colors = ['steelblue' if m == 'SimpleNN' else 'coral' for m in models]
    bars = ax.barh(models, ranks, color=colors, edgecolor='black', linewidth=1.5)
    
    # Add rank values
    for i, (model, rank) in enumerate(zip(models, ranks)):
        ax.text(rank, i, f' {rank:.2f}', va='center', fontsize=11, fontweight='bold')
    
    ax.set_xlabel('Average Rank', fontsize=12, fontweight='bold')
    ax.set_title('Critical Difference Diagram (Statistical Ranking)', fontsize=14, fontweight='bold')
    ax.set_xlim([0, 3])
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")


print("Plotting functions defined.")

In [ ]:
# Generate all required plots
print("Generating plots...\n")

# Graph 1: Accuracy vs Distance
plot_accuracy_vs_distance(
    accuracy_results, 
    statistical_tests, 
    PLOTS_DIR / "accuracy_vs_distance.png"
)

# Graph 2: Pareto Frontier
plot_pareto_frontier(
    accuracy_results,
    inference_benchmarks,
    PLOTS_DIR / "pareto_frontier.png"
)

# Graph 3: Convergence Curves
plot_convergence_curves(
    training_convergence,
    PLOTS_DIR / "convergence_curves.png"
)

# Graph 4: Throughput vs Batch Size
plot_throughput_vs_batchsize(
    inference_benchmarks,
    PLOTS_DIR / "throughput_vs_batchsize.png"
)

# Graph 5: Critical Difference
plot_critical_difference(
    accuracy_results,
    statistical_tests,
    PLOTS_DIR / "critical_difference.png"
)

print("\n✓ All plots generated")

## 13. Create Summary Table

In [ ]:
# Create comprehensive summary table
summary_data = []

for d in sorted(DISTANCES):
    if d not in accuracy_results or d not in inference_benchmarks:
        continue
    
    # SimpleNN row
    if 'simplenn' in accuracy_results[d] and 'simplenn' in inference_benchmarks[d]:
        simplenn_acc = accuracy_results[d]['simplenn']
        simplenn_inf = inference_benchmarks[d]['simplenn']
        batch_size_64 = simplenn_inf.get(64, {})
        
        p_value = statistical_tests.get(d, {}).get('wilcoxon', {}).get('p_value', np.nan)
        effect_size = statistical_tests.get(d, {}).get('cohens_d', np.nan)
        
        summary_data.append({
            'Distance': d,
            'Model': 'SimpleNN',
            'Params': simplenn_info.get(d, {}).get('num_parameters', 0),
            'Accuracy (mean±std)': f"{simplenn_acc['accuracy_mean']:.4f}±{simplenn_acc['accuracy_std']:.4f}",
            'F1 (mean±std)': f"{simplenn_acc['f1_mean']:.4f}±{simplenn_acc['f1_std']:.4f}",
            'Latency (μs, mean±std)': f"{batch_size_64.get('latency_per_sample_us', 0):.2f}±{batch_size_64.get('std_time_seconds', 0)*1e6/len(shared_test_data[d]['labels']):.2f}" if batch_size_64 else "N/A",
            'Throughput (samples/s)': f"{batch_size_64.get('throughput_samples_per_sec', 0):.0f}" if batch_size_64 else "N/A",
            'p-value vs SimpleNN': '—',
            'Effect Size': '—'
        })
    
    # GraphSAGE row
    if 'graphsage' in accuracy_results[d] and 'graphsage' in inference_benchmarks[d]:
        graphsage_acc = accuracy_results[d]['graphsage']
        graphsage_inf = inference_benchmarks[d]['graphsage']
        batch_size_64 = graphsage_inf.get(64, {})
        
        p_value = statistical_tests.get(d, {}).get('wilcoxon', {}).get('p_value', np.nan)
        effect_size = statistical_tests.get(d, {}).get('cohens_d', np.nan)
        
        summary_data.append({
            'Distance': d,
            'Model': 'GraphSAGE',
            'Params': graphsage_info.get(d, {}).get('num_parameters', 0),
            'Accuracy (mean±std)': f"{graphsage_acc['accuracy_mean']:.4f}±{graphsage_acc['accuracy_std']:.4f}",
            'F1 (mean±std)': f"{graphsage_acc['f1_mean']:.4f}±{graphsage_acc['f1_std']:.4f}",
            'Latency (μs, mean±std)': f"{batch_size_64.get('latency_per_sample_us', 0):.2f}±{batch_size_64.get('std_time_seconds', 0)*1e6/len(shared_test_data[d]['labels']):.2f}" if batch_size_64 else "N/A",
            'Throughput (samples/s)': f"{batch_size_64.get('throughput_samples_per_sec', 0):.0f}" if batch_size_64 else "N/A",
            'p-value vs SimpleNN': f"{p_value:.4f}" if not np.isnan(p_value) else "N/A",
            'Effect Size': f"{effect_size:.3f}" if not np.isnan(effect_size) else "N/A"
        })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(OUTPUT_RESULTS_DIR / "summary_table.csv", index=False)

print("Summary Table:")
print(summary_df.to_string(index=False))
print(f"\n✓ Summary table saved to {OUTPUT_RESULTS_DIR / 'summary_table.csv'}")

## 14. Save Results and Generate Final Report

In [ ]:
# Prepare all results for saving
comparison_results = {
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'distances': DISTANCES,
        'test_samples_per_distance': TEST_SAMPLES_PER_DISTANCE,
        'num_benchmark_runs': NUM_BENCHMARK_RUNS,
        'num_accuracy_runs': NUM_ACCURACY_RUNS,
        'significance_level': ALPHA,
        'seed': SEED
    },
    'parameter_comparison': param_comparison,
    'accuracy_results': {
        str(d): {
            'simplenn': {
                'accuracy_mean': float(accuracy_results[d]['simplenn']['accuracy_mean']),
                'accuracy_std': float(accuracy_results[d]['simplenn']['accuracy_std']),
                'accuracy_ci_95': float(accuracy_results[d]['simplenn']['accuracy_ci_95']),
                'f1_mean': float(accuracy_results[d]['simplenn']['f1_mean']),
                'f1_std': float(accuracy_results[d]['simplenn']['f1_std']),
                'precision_mean': float(accuracy_results[d]['simplenn']['precision_mean']),
                'recall_mean': float(accuracy_results[d]['simplenn']['recall_mean'])
            },
            'graphsage': {
                'accuracy_mean': float(accuracy_results[d]['graphsage']['accuracy_mean']),
                'accuracy_std': float(accuracy_results[d]['graphsage']['accuracy_std']),
                'accuracy_ci_95': float(accuracy_results[d]['graphsage']['accuracy_ci_95']),
                'f1_mean': float(accuracy_results[d]['graphsage']['f1_mean']),
                'f1_std': float(accuracy_results[d]['graphsage']['f1_std']),
                'precision_mean': float(accuracy_results[d]['graphsage']['precision_mean']),
                'recall_mean': float(accuracy_results[d]['graphsage']['recall_mean'])
            }
        } for d in accuracy_results.keys()
    },
    'model_info': {
        'simplenn': {str(d): simplenn_info[d] for d in simplenn_info.keys()},
        'graphsage': {str(d): graphsage_info[d] for d in graphsage_info.keys()}
    }
}

# Save comparison results
with open(OUTPUT_RESULTS_DIR / "comparison_results.json", 'w') as f:
    json.dump(comparison_results, f, indent=2)

# Save inference benchmarks (simplified for JSON)
inference_benchmarks_json = {}
for d in inference_benchmarks.keys():
    inference_benchmarks_json[str(d)] = {}
    for model_type in ['simplenn', 'graphsage']:
        if model_type in inference_benchmarks[d]:
            inference_benchmarks_json[str(d)][model_type] = {
                str(bs): {
                    'throughput_samples_per_sec': float(inference_benchmarks[d][model_type][bs]['throughput_samples_per_sec']),
                    'latency_per_sample_us': float(inference_benchmarks[d][model_type][bs]['latency_per_sample_us']),
                    'mean_time_seconds': float(inference_benchmarks[d][model_type][bs]['mean_time_seconds']),
                    'std_time_seconds': float(inference_benchmarks[d][model_type][bs]['std_time_seconds'])
                } for bs in inference_benchmarks[d][model_type].keys()
            }

with open(OUTPUT_RESULTS_DIR / "inference_benchmarks.json", 'w') as f:
    json.dump(inference_benchmarks_json, f, indent=2)

# Save statistical tests
# Convert numpy types to native Python types for JSON
def convert_to_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_json_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    else:
        return obj

statistical_tests_json = convert_to_json_serializable(statistical_tests)
with open(OUTPUT_RESULTS_DIR / "statistical_tests.json", 'w') as f:
    json.dump(statistical_tests_json, f, indent=2)

print("✓ All results saved to JSON files")

In [ ]:
# Generate final summary report
report_lines = []
report_lines.append("=" * 80)
report_lines.append("NEURAL NETWORK vs GRAPH NEURAL NETWORK COMPARISON REPORT")
report_lines.append("=" * 80)
report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append("")

# Overall statistical test results
if 'overall' in statistical_tests:
    overall = statistical_tests['overall']
    wilcoxon_result = overall['wilcoxon']
    cohens_d_val = overall['cohens_d']
    
    report_lines.append("OVERALL STATISTICAL ANALYSIS")
    report_lines.append("-" * 80)
    report_lines.append(f"Wilcoxon Signed-Rank Test:")
    report_lines.append(f"  Statistic: {wilcoxon_result['statistic']:.4f}")
    report_lines.append(f"  p-value: {wilcoxon_result['p_value']:.6f}")
    report_lines.append(f"  Significant (α={ALPHA}): {'YES' if wilcoxon_result['significant'] else 'NO'}")
    report_lines.append(f"  Effect Size (Cohen's d): {cohens_d_val:.4f} ({overall['effect_size_interpretation']})")
    report_lines.append("")

# Per-distance summary
report_lines.append("PER-DISTANCE SUMMARY")
report_lines.append("-" * 80)
for d in sorted(accuracy_results.keys()):
    if 'simplenn' not in accuracy_results[d] or 'graphsage' not in accuracy_results[d]:
        continue
    
    simplenn_acc = accuracy_results[d]['simplenn']['accuracy_mean']
    graphsage_acc = accuracy_results[d]['graphsage']['accuracy_mean']
    
    if d in statistical_tests:
        p_val = statistical_tests[d]['wilcoxon']['p_value']
        sig = statistical_tests[d]['wilcoxon']['significant']
        effect = statistical_tests[d]['cohens_d']
    else:
        p_val = np.nan
        sig = False
        effect = np.nan
    
    report_lines.append(f"Distance d={d}:")
    report_lines.append(f"  SimpleNN: {simplenn_acc:.4f} ± {accuracy_results[d]['simplenn']['accuracy_std']:.4f}")
    report_lines.append(f"  GraphSAGE: {graphsage_acc:.4f} ± {accuracy_results[d]['graphsage']['accuracy_std']:.4f}")
    report_lines.append(f"  Difference: {graphsage_acc - simplenn_acc:+.4f}")
    report_lines.append(f"  p-value: {p_val:.6f} {'(significant)' if sig else '(not significant)'}")
    report_lines.append(f"  Effect size: {effect:.4f}")
    report_lines.append("")

# Final verdict
report_lines.append("FINAL VERDICT")
report_lines.append("-" * 80)

if 'overall' in statistical_tests:
    overall_wilcoxon = statistical_tests['overall']['wilcoxon']
    overall_cohens_d = statistical_tests['overall']['cohens_d']
    
    # Calculate average accuracies
    simplenn_avg_acc = np.mean([accuracy_results[d]['simplenn']['accuracy_mean'] for d in accuracy_results.keys() if 'simplenn' in accuracy_results[d]])
    graphsage_avg_acc = np.mean([accuracy_results[d]['graphsage']['accuracy_mean'] for d in accuracy_results.keys() if 'graphsage' in accuracy_results[d]])
    
    # Calculate average throughput (batch size 64)
    simplenn_throughputs = []
    graphsage_throughputs = []
    for d in inference_benchmarks.keys():
        if 'simplenn' in inference_benchmarks[d] and 64 in inference_benchmarks[d]['simplenn']:
            simplenn_throughputs.append(inference_benchmarks[d]['simplenn'][64]['throughput_samples_per_sec'])
        if 'graphsage' in inference_benchmarks[d] and 64 in inference_benchmarks[d]['graphsage']:
            graphsage_throughputs.append(inference_benchmarks[d]['graphsage'][64]['throughput_samples_per_sec'])
    
    simplenn_avg_throughput = np.mean(simplenn_throughputs) if simplenn_throughputs else 0
    graphsage_avg_throughput = np.mean(graphsage_throughputs) if graphsage_throughputs else 0
    
    verdict = f"""GraphSAGE achieved {graphsage_avg_acc:.4f} accuracy (mean ± std) vs SimpleNN's {simplenn_avg_acc:.4f} accuracy (mean ± std) across distances d=3 to d=13. The Wilcoxon signed-rank test showed this difference {'WAS' if overall_wilcoxon['significant'] else 'WAS NOT'} statistically significant (p = {overall_wilcoxon['p_value']:.6f}, effect size = {overall_cohens_d:.4f}). However, SimpleNN achieved {simplenn_avg_throughput/graphsage_avg_throughput:.2f}x faster inference throughput ({simplenn_avg_throughput:.0f} samples/s vs {graphsage_avg_throughput:.0f} samples/s). For real-time QEC applications requiring low latency, {'SimpleNN' if simplenn_avg_throughput > graphsage_avg_throughput else 'GraphSAGE'} is recommended based on the Pareto frontier analysis."""
    
    report_lines.append(verdict)

report_lines.append("")
report_lines.append("=" * 80)

# Save report
report_text = "\n".join(report_lines)
with open(OUTPUT_RESULTS_DIR / "summary_report.txt", 'w') as f:
    f.write(report_text)

print("\n" + report_text)
print(f"\n✓ Summary report saved to {OUTPUT_RESULTS_DIR / 'summary_report.txt'}")